# Creating the SQL Database

The SQL database won't be the main part of this project, however I think it's good to have it. Once we have cleaned all the datasets individually and, in theory, we won't further edit them, it's a great opportunity to create a SQL database. It'll be as having a picture of our datasets all together. It'll help us visualize the structure of the database and we'll be able to query it when needed. 

The main objective now is just creating a big dataset with all the needed data and jump to Tableau to start visualizing it and answer our questions. Even though, that could be done from python, as I said I think it's good to have it in SQL ready to use it if we come up with new questions during the visualizations. I prefer to use python if I need to edit our dataframes and SQL if I have to answer questions or do joins.

## Importing Libraries

In [1]:
import pandas as pd

from sqlalchemy import create_engine 
from sqlalchemy import text 
import getpass

## Creating the Engine and Database

In [4]:
password = getpass.getpass()

connection_string = 'mysql+pymysql://root:' + password + '@localhost/'
engine = create_engine(connection_string)
connection = engine.connect()
query = text("""CREATE DATABASE IF NOT EXISTS FindFilms;""")
connection.execute(query)

## Creating and populating the tables in SQL

We already created the database. Now, we create a new connection, in this case, to connect with the database we just created.

In [5]:
password = getpass.getpass()

bd = "FindFilms"
connection_string = 'mysql+pymysql://root:' + password + '@localhost/'+bd
engine = create_engine(connection_string)
connection = engine.connect()

We need to load our datasets.

In [ ]:
def load_datasets(name):
    df_loc = f"../data/processed/{name}.csv"
    df = pd.read_csv(df_loc)
    return df

datasets_list = ["links", "ratings", "genome_tags", "genome_scores",
                "title_basics", "title_genres", "title_ratings", "bechdel", "tmdb_financial"]

datasets = {}

for name in datasets_list:
    datasets[name] = load_datasets(name)

In [13]:
datasets.keys()

dict_keys(['links', 'ratings', 'genome_tags', 'genome_scores', 'title_basics', 'title_genres', 'title_ratings', 'bechdel', 'tmdb_financial'])

We send our datasets to SQL

In [14]:
def datasets_to_sql(table_name, df):
    df.to_sql(
        name = table_name,
        con = engine,
        schema = "FindFilms",
        if_exists = "replace",
        index = False
    )

    return f"{table_name} sent to SQL"


for name, df in datasets.items():
    print(datasets_to_sql(name, df))

links sent to SQL
ratings sent to SQL
genome_tags sent to SQL
genome_scores sent to SQL
title_basics sent to SQL
title_genres sent to SQL
title_ratings sent to SQL
bechdel sent to SQL
tmdb_financial sent to SQL


In [27]:
df_loc = "../data/processed/bechdel.csv"
bechdel = pd.read_csv(df_loc)
datasets_to_sql("bechdel", bechdel)

'bechdel sent to SQL'

In [31]:
df_loc = "../data/processed/tmdb_financial.csv"
tmdb_financial = pd.read_csv(df_loc)
datasets_to_sql("tmdb_financial", tmdb_financial)

'tmdb_financial sent to SQL'

### Primary keys and foreign keys

I had an error because they were importing the imbd_id as text instead of VARCHAR.

In [ ]:
query = text("""ALTER TABLE links
                MODIFY imdb_id VARCHAR(20)""")
connection.execute(query)

query = text("""ALTER TABLE title_basics
                MODIFY imdb_id VARCHAR(20)""")
connection.execute(query)

query = text("""ALTER TABLE title_genres
                MODIFY imdb_id VARCHAR(20)""")
connection.execute(query)

query = text("""ALTER TABLE title_ratings
                MODIFY imdb_id VARCHAR(20)""")
connection.execute(query)

query = text("""ALTER TABLE bechdel
                MODIFY imdb_id VARCHAR(20)""")
connection.execute(query)

In [ ]:
query = text("""ALTER TABLE links
                ADD INDEX idx_movie_id (movie_id)
""")
connection.execute(query)

query = text("""ALTER TABLE links
                ADD INDEX idx_tmdb_id (tmdb_id)
""")
connection.execute(query)

With all the tables created and populated we need to assign primary keys and foreign keys.

In [ ]:
# Assign the primary keys
query = text("""ALTER TABLE `links`
                ADD PRIMARY KEY(`imdb_id`)""")
connection.execute(query)

query = text("""ALTER TABLE `genome_tags`
                ADD PRIMARY KEY(`tag_id`)""")
connection.execute(query)

query = text("""ALTER TABLE `title_basics`
                ADD PRIMARY KEY(`imdb_id`)""")
connection.execute(query)

query = text("""ALTER TABLE `title_ratings`
                ADD PRIMARY KEY(`imdb_id`)""")
connection.execute(query)

query = text("""ALTER TABLE `bechdel`
                ADD PRIMARY KEY(`imdb_id`)""")
connection.execute(query)

query = text("""ALTER TABLE `tmdb_financial`
                ADD PRIMARY KEY(`tmdb_id`)""")
connection.execute(query)

# Altering tables to connect the foreign keys
query = text("""ALTER TABLE `ratings`
                ADD CONSTRAINT `fk_movie_id_ratings`
                FOREIGN KEY(`movie_id`) REFERENCES `links`(`movie_id`)""")
connection.execute(query)

query = text("""ALTER TABLE `genome_scores`
                ADD CONSTRAINT `fk_movie_id_scores`
                FOREIGN KEY(`movie_id`) REFERENCES `links`(`movie_id`),
             
                ADD CONSTRAINT `fk_tag_id_scores`
                FOREIGN KEY(`tag_id`) REFERENCES `genome_tags`(`tag_id`)""")
connection.execute(query)

query = text("""ALTER TABLE `title_basics`
                ADD CONSTRAINT `fk_imdb_id_basics`
                FOREIGN KEY(`imdb_id`) REFERENCES `links`(`imdb_id`)""")
connection.execute(query)

query = text("""ALTER TABLE `title_genres`
                ADD CONSTRAINT `fk_imdb_id_genres`
                FOREIGN KEY(`imdb_id`) REFERENCES `title_basics`(`imdb_id`)""")
connection.execute(query)

query = text("""ALTER TABLE `title_ratings`
                ADD CONSTRAINT `fk_imdb_id_ratings`
                FOREIGN KEY(`imdb_id`) REFERENCES `title_basics`(`imdb_id`)""")
connection.execute(query)

query = text("""ALTER TABLE `bechdel`
                ADD CONSTRAINT `fk_imdb_id_bechdel`
                FOREIGN KEY(`imdb_id`) REFERENCES `links`(`imdb_id`)""")
connection.execute(query)

query = text("""ALTER TABLE `tmdb_financial`
                ADD CONSTRAINT `fk_imdb_id_financial`
                FOREIGN KEY(`tmdb_id`) REFERENCES `links`(`tmdb_id`)""")
connection.execute(query)